# Học và kiểm chứng phần A: baseline GoEmotions

Đọc từ trên xuống. Mỗi cell nối một khái niệm NLP với dữ liệu hoặc kết quả thật.
Notebook chỉ mở train/validation và model đã fit; test dành cho protocol cuối.

Nếu chưa có model: chạy `python -m scripts.run_baseline`, tiếp đến `--variant balanced`,
rồi `python -m scripts.analyze_baseline` từ gốc repo. Cài `requirements-notebook.txt`
để chạy notebook bằng Jupyter hoặc VS Code. Phần triển khai chính nằm trong script,
nên nhóm có thể dùng cùng mã và metric khi tổng hợp A/B/C.

In [1]:
from pathlib import Path
import sys
import json
import joblib
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.data import load_goemotions, multi_hot
from src.baseline import load_run_metadata, load_aligned_scores, load_thresholds
from src.metrics import evaluate_multilabel
print('Python:', sys.version.split()[0])

Python: 3.13.9


## 1. Dữ liệu và 28 nhãn

Một câu có thể có nhiều cảm xúc. Không dùng argmax để ép chọn duy nhất một nhãn.
Dữ liệu simplified có 43.410 train và 5.426 validation. Revision/SHA-256 được kiểm
trong `src/data.py`; `id` dùng để ghép kết quả các mô hình.

In [2]:
frames, labels, manifest = load_goemotions(ROOT, write_metadata=False, splits=('train', 'validation'))
train, val = frames['train'], frames['validation']
y_train = multi_hot(train['labels'].tolist(), len(labels))
y_val = multi_hot(val['labels'].tolist(), len(labels))
print('Y train:', y_train.shape, '| Y validation:', y_val.shape)
pd.DataFrame({'label_id': range(len(labels)), 'label': labels,
              'train_positive': y_train.sum(axis=0), 'val_positive': y_val.sum(axis=0)}).head(8)

Y train: (43410, 28) | Y validation: (5426, 28)


,label_id,label,train_positive,val_positive
0,0,admiration,4130,488
1,1,amusement,2328,303
2,2,anger,1567,195
3,3,annoyance,2470,303
4,4,approval,2939,397
5,5,caring,1087,153
6,6,confusion,1368,152
7,7,curiosity,2191,248


## 2. Multi-hot có nghĩa gì?

Hai nhãn admiration và gratitude được giữ đồng thời. Neutral là một cột riêng và
được giữ nguyên như dữ liệu nguồn. Ma trận sau chỉ là ví dụ minh họa.

In [3]:
example = multi_hot([[labels.index('admiration'), labels.index('gratitude')],
                     [labels.index('neutral')]], len(labels))
pd.DataFrame(example, columns=labels)[['admiration', 'gratitude', 'neutral']]

,admiration,gratitude,neutral
0,1,1,0
1,0,0,1


## 3. Nạp pipeline đã fit trên train

Pipeline gồm TF-IDF và 28 Logistic Regression nhị phân. Model và scores được
kiểm hash để tránh trộn kết quả từ các lần chạy. TF-IDF chỉ fit trên train.
Lệnh `transform`/`predict_proba` trên validation không thêm từ vào từ vựng.

In [4]:
folder = ROOT / 'data/processed/baseline/full'
metadata = load_run_metadata(folder, 'standard', labels)
model = joblib.load(folder / 'model.joblib')
scores = load_aligned_scores(folder / 'validation_scores.npz', val['id'].tolist(), labels)
print('So dac trung TF-IDF:', len(model.named_steps['tfidf'].vocabulary_))
print('So Logistic Regression:', len(model.named_steps['classifier'].estimators_))
print('Scores validation:', scores.shape)
print('So vong lap toi uu lon nhat:', max(metadata['optimizer_iterations']))

So dac trung TF-IDF: 58338
So Logistic Regression: 28
Scores validation: (5426, 28)
So vong lap toi uu lon nhat: 12


## 4. Tokenization mặc định và hạn chế

Baseline này dùng từ/cặp từ, lowercase và token pattern mặc định của scikit-learn.
Nó bỏ dấu câu, emoji và token một ký tự. `don't` có thể thành `don`; vì vậy chưa
thể nói baseline hiểu tốt mọi phủ định hoặc mỉa mai. Dùng cell sau để quan sát.

In [5]:
analyzer = model.named_steps['tfidf'].build_analyzer()
analyzer("I don't like this! 😢")

['don', 'like', 'this', 'don like', 'like this']

## 5. Điểm khác nhãn cuối như thế nào?

Chọn một ví dụ pride bị bỏ sót ở ngưỡng 0,5. Điểm các nhãn độc lập và không cần
cộng lại bằng 1. Bạn có thể đổi ID trong cell để xem một câu khác.

In [6]:
sample_id = 'eczwil0'
i = val.index[val['id'] == sample_id][0]
print(val.loc[i, 'text'])
print('Nhan that:', [labels[j] for j in val.loc[i, 'labels']])
view = pd.DataFrame({'label': labels, 'score': scores[i], 'predicted_at_0_5': scores[i] >= 0.5})
view.sort_values('score', ascending=False).head(6)

I am so proud of this community.
Nhan that: ['pride']


,label,score,predicted_at_0_5
21,pride,0.377635,False
0,admiration,0.359615,False
27,neutral,0.074781,False
17,joy,0.058782,False
25,sadness,0.042666,False
7,curiosity,0.039619,False


## 6. Micro/Macro-F1 và Hamming Loss

Macro-F1 lấy trung bình đều 28 F1, Micro-F1 cộng TP/FP/FN trước khi tính.
Hamming Loss là tỷ lệ các ô nhãn sai; cần đọc cùng Recall và F1 nhãn hiếm.
Hàm metric còn lưu TP/FP/FN/TN để tự kiểm công thức.

In [7]:
metric = evaluate_multilabel(y_val, scores, labels)
pd.Series({key: metric[key] for key in ('macro_f1', 'micro_f1', 'micro_precision',
                                     'micro_recall', 'hamming_loss', 'empty_prediction_count')})

macro_f1                     0.202457
micro_f1                     0.375987
micro_precision              0.725358
micro_recall                 0.253762
hamming_loss                 0.035372
empty_prediction_count    3271.000000
dtype: float64

## 7. So sánh ngưỡng 0,5, ngưỡng chung và ngưỡng riêng

Tất cả lựa chọn ngưỡng được thực hiện trên validation. Hàng chọn ngưỡng đo trên
chính tập dùng chọn ngưỡng, nên có thể lạc quan. Test sau freeze mới xác nhận.
Class weighting là biến thể model khác, phải có ngưỡng của chính model đó.

In [8]:
rows = []
for variant, location in [('standard', ROOT/'data/processed/baseline/full'),
                          ('balanced', ROOT/'data/processed/baseline/balanced/full')]:
    saved = json.loads((location/'analysis_validation.json').read_text(encoding='utf-8'))
    for mode, key in [('fixed', 'fixed_0_5'), ('global', 'global_on_validation'),
                      ('tuned', 'tuned_on_validation')]:
        item = saved[key]
        rows.append({'variant': variant, 'threshold_mode': mode,
                     **{key: item[key] for key in ('macro_f1', 'micro_f1', 'hamming_loss')}})
pd.DataFrame(rows)

,variant,threshold_mode,macro_f1,micro_f1,hamming_loss
0,standard,fixed,0.202457,0.375987,0.035372
1,standard,global,0.409351,0.510020,0.054394
2,standard,tuned,0.439074,0.542750,0.043297
3,balanced,fixed,0.456249,0.509942,0.053209
4,balanced,global,0.465984,0.517609,0.047332
5,balanced,tuned,0.490054,0.546680,0.044264


## 8. Nhãn hiếm và cặp lỗi

Năm nhãn hiếm được xác định từ train. Bảng FN(A)+FP(B) chỉ đếm hai lỗi trong
cùng câu; khác với nhãn thật đồng xuất hiện. Một câu có thể góp nhiều cặp.

In [9]:
per_label = pd.read_csv(folder/'per_label_validation.csv')
per_label.nsmallest(5, 'train_support')[['label', 'train_support', 'validation_support',
                                      'f1_fixed', 'f1_global_val', 'f1_tuned_val']]

,label,train_support,validation_support,f1_fixed,f1_global_val,f1_tuned_val
16,grief,77,13,0.000000,0.000000,0.000000
21,pride,111,15,0.000000,0.333333,0.333333
23,relief,153,18,0.000000,0.000000,0.105263
19,nervousness,164,21,0.000000,0.086957,0.090909
12,embarrassment,303,35,0.108108,0.105263,0.363636


In [10]:
pairs = pd.read_csv(folder/'label_error_pairs_validation.csv')
pairs[(pairs['missed_label'] != 'neutral') & (pairs['extra_label'] != 'neutral')].head(6)

,missed_label,extra_label,count,missed_label_total_fn,fraction_of_fn,example_id
13,admiration,love,10,311,0.032154,ee9xgzw
17,joy,love,8,147,0.054422,ed0ht04
19,joy,admiration,7,147,0.047619,edopy68
25,approval,love,5,384,0.013021,ed031mb
26,gratitude,joy,5,81,0.061728,eexk0cl
29,approval,admiration,4,384,0.010417,edcmnk3


## 9. Tự nhập văn bản, xem score và ngưỡng

Đổi câu tiếng Anh bên dưới. Dùng model balanced và ngưỡng riêng đã chọn từ
validation. Demo cuối của nhóm vẫn phải dùng C tốt nhất theo yêu cầu cô.

In [11]:
text = 'I am so proud of you!'
balanced_folder = ROOT/'data/processed/baseline/balanced/full'
balanced_meta = load_run_metadata(balanced_folder, 'balanced', labels)
balanced_model = joblib.load(balanced_folder/'model.joblib')
thresholds = np.broadcast_to(load_thresholds(balanced_folder, balanced_meta, 'tuned'), (len(labels),))
new_scores = balanced_model.predict_proba([text])[0]
pd.DataFrame({'label': labels, 'score': new_scores, 'threshold': thresholds,
              'predicted': new_scores >= thresholds}).sort_values('score', ascending=False).head(6)

,label,score,threshold,predicted
21,pride,0.999969,0.80,True
0,admiration,0.912555,0.55,True
17,joy,0.283661,0.70,False
15,gratitude,0.231854,0.75,False
13,excitement,0.200819,0.60,False
7,curiosity,0.167609,0.70,False


## 10. Việc bạn cần tự giải thích

1. Vì sao số feature là 58.338 còn số nhãn là 28?
2. Vì sao không fit TF-IDF trên validation/test?
3. Vì sao weighting có thể tăng Recall và cả FP?
4. Vì sao score pride 0,3776 không vượt ngưỡng 0,5?
5. Vì sao điểm tuned-validation chưa phải cải thiện trên test?
6. Vì sao bảng cặp lỗi không phải bảng nhãn thật đồng xuất hiện?

Xem `docs/BASELINE.md` để chạy lại và `reports/BASELINE_RESULTS.md` để dùng số thật.
Nguồn: [bài báo GoEmotions](https://aclanthology.org/2020.acl-main.372/),
[TF-IDF](https://scikit-learn.org/1.7/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html),
[One-vs-Rest](https://scikit-learn.org/1.7/modules/generated/sklearn.multiclass.OneVsRestClassifier.html),
[chọn ngưỡng](https://scikit-learn.org/1.7/modules/classification_threshold.html).

## 11. Đọc kết quả test sau khi khóa cấu hình

Cell dưới chỉ đọc kết quả đã lưu, không học lại và không chọn ngưỡng bằng test.
Bảng có sáu cấu hình đã quyết định từ validation. Đừng chọn lại model chỉ vì một hàng test cao hơn.
Nếu chưa có file, tiếp tục pipeline nhóm theo `docs/CHAY_THUC_NGHIEM.md`; không điền số dự kiến.

In [12]:
from src.data import sha256, REVISION
test_path = ROOT / 'data/processed/baseline/final/final_results.json'
if not test_path.exists():
    print('Chưa có kết quả test A sau khóa protocol. Không suy ra từ validation.')
else:
    final = json.loads(test_path.read_text(encoding='utf-8'))
    protocol_path = ROOT / 'data/processed/baseline/final_protocol.json'
    if final['protocol_sha256'] != sha256(protocol_path) or final['data_revision'] != REVISION:
        raise ValueError('Kết quả test không khớp protocol hoặc revision dữ liệu')
    print('Cấu hình chọn trên validation:', final['selected_on_validation'])
    test_rows = [dict(configuration=r['name'], **{k: r['metrics'][k] for k in
                 ['macro_f1', 'micro_f1', 'macro_precision', 'macro_recall',
                  'micro_precision', 'micro_recall', 'hamming_loss']}) for r in final['results']]
    display(pd.DataFrame(test_rows))

Cấu hình chọn trên validation: balanced_tuned


,configuration,macro_f1,micro_f1,macro_precision,macro_recall,micro_precision,micro_recall,hamming_loss
0,standard_fixed,0.196339,0.379958,0.612794,0.139578,0.738258,0.255807,0.034773
1,standard_global,0.409569,0.504745,0.463092,0.447578,0.402519,0.676568,0.055299
2,standard_tuned,0.413426,0.532992,0.438381,0.436140,0.474365,0.608153,0.044388
3,balanced_fixed,0.444052,0.502364,0.377677,0.569586,0.404335,0.663138,0.054720
4,balanced_global,0.453027,0.515671,0.402946,0.536328,0.444737,0.613525,0.048001
5,balanced_tuned,0.449264,0.527739,0.437236,0.488342,0.456136,0.626007,0.046665
